venue comparison

In [ ]:
import pandas as pd

# -------------------------------
# Load new normalized file
# -------------------------------
df = pd.read_csv("classification_framework_final.csv")

# -------------------------------
# Venue mapping
# -------------------------------
def map_conference_to_venue(conf_str):
    s = str(conf_str).strip().upper()

    if "USA" in s:
        return "DFRWS USA"
    if "EU" in s or "EUROPE" in s:
        return "DFRWS EU"
    if "APAC" in s or "ASIA" in s:
        return "DFRWS APAC"

    return "Unknown"

# -------------------------------
# Clean columns
# -------------------------------
df["venue"] = df["conference"].apply(map_conference_to_venue)
df["category"] = df["category"].astype(str).str.strip()

# Remove missing / unknown rows
clean_df = df[
    (df["venue"] != "Unknown") &
    (df["category"].notna()) &
    (df["category"] != "") &
    (df["category"].str.lower() != "nan")
].copy()

# -------------------------------
# Venue × Primary Domain matrix
# -------------------------------
venue_domain_matrix = pd.crosstab(
    clean_df["venue"],
    clean_df["category"]
).astype(int)

# Order columns by total count
col_order = venue_domain_matrix.sum(axis=0).sort_values(ascending=False).index
venue_domain_matrix = venue_domain_matrix[col_order]

# Order rows
row_order = [v for v in ["DFRWS USA", "DFRWS EU", "DFRWS APAC"] if v in venue_domain_matrix.index]
venue_domain_matrix = venue_domain_matrix.reindex(row_order)

# -------------------------------
# Add stats
# -------------------------------
core = venue_domain_matrix.copy()

venue_domain_matrix["TOTAL"] = core.sum(axis=1)
venue_domain_matrix["MEDIAN"] = core.median(axis=1).round(2)

means = core.mean(axis=1)
stds = core.std(axis=1, ddof=1)
venue_domain_matrix["CV"] = (stds / means.replace(0, pd.NA)).fillna(0).round(3)

# -------------------------------
# Save output
# -------------------------------
out_path = "venue_category_matrix.csv"
venue_domain_matrix.to_csv(out_path)

print(venue_domain_matrix)
print("\nSaved:", out_path)
print("\nRows used:", len(clean_df), "out of", len(df))

Categories total count

In [ ]:
import pandas as pd

# -------------------------------
# Load file
# -------------------------------
df = pd.read_csv("classification_framework_final.csv")

# -------------------------------
# Clean columns
# -------------------------------
df["category"] = df["category"].astype(str).str.strip()
df["subcategory"] = df["subcategory"].astype(str).str.strip()

clean_df = df[
    (df["category"].notna()) &
    (df["subcategory"].notna()) &
    (df["category"] != "") &
    (df["subcategory"] != "") &
    (df["category"].str.lower() != "nan") &
    (df["subcategory"].str.lower() != "nan")
].copy()

# -------------------------------
# Category + Subcategory counts
# -------------------------------
summary = (
    clean_df
    .groupby(["category", "subcategory"])
    .size()
    .reset_index(name="count")
)

# -------------------------------
# Add percentages
# -------------------------------
total_papers = len(clean_df)

summary["percentage_total"] = (
    summary["count"] / total_papers * 100
).round(2)

summary["percentage_within_category"] = (
    summary["count"] /
    summary.groupby("category")["count"].transform("sum") * 100
).round(2)

# -------------------------------
# Add category totals
# -------------------------------
summary["category_total"] = summary.groupby("category")["count"].transform("sum")

# -------------------------------
# Sort
# -------------------------------
summary = summary.sort_values(
    by=["category_total", "category", "count"],
    ascending=[False, True, False]
)

# Optional: reorder columns
summary = summary[
    [
        "category",
        "subcategory",
        "count",
        "category_total",
        "percentage_within_category",
        "percentage_total"
    ]
]

# -------------------------------
# Save output
# -------------------------------
out_path = "category_subcategory_counts_percentages.csv"
summary.to_csv(out_path, index=False)

print(summary.to_string(index=False))
print("\nSaved:", out_path)
print("Rows used:", len(clean_df), "out of", len(df))

Top subcategories

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------
# Load data
# -------------------------------
df = pd.read_csv("classification_framework_final.csv")

df["published_year"] = pd.to_numeric(
    df["published_year"],
    errors="coerce"
)

df = df[
    (df["published_year"] >= 2002) &
    (df["published_year"] <= 2025)
]

# -------------------------------
# Select Top 15 Subcategories
# -------------------------------
top15 = (
    df["subcategory"]
    .dropna()
    .value_counts()
    .head(15)
    .index
    .tolist()
)

print("\nTop 15 Subcategories:")
for i, s in enumerate(top15, 1):
    print(f"{i:2d}. {s}")

# -------------------------------
# Create Year × Subcategory Matrix
# -------------------------------
plot_df = (
    df[df["subcategory"].isin(top15)]
    .groupby(["published_year", "subcategory"])
    .size()
    .unstack(fill_value=0)
    .reindex(range(2002, 2026), fill_value=0)
)

# Preserve ranking order
plot_df = plot_df.reindex(
    columns=top15,
    fill_value=0
)

# -------------------------------
# Save underlying data
# -------------------------------
plot_df.to_csv(
    "top15_subcategories_over_time.csv"
)

# -------------------------------
# Plot
# -------------------------------
plt.figure(figsize=(14, 8))

for col in plot_df.columns:
    plt.plot(
        plot_df.index,
        plot_df[col],
        marker="o",
        linewidth=2,
        label=col
    )

plt.xlabel("Published Year")
plt.ylabel("Number of Papers")
plt.title(
    "Top 15 Subcategories Over Time (2002–2025)"
)

plt.xticks(
    range(2002, 2026),
    rotation=45
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=9
)

plt.tight_layout()

# -------------------------------
# Save Figure
# -------------------------------
plt.savefig(
    "top15_subcategories_over_time.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(" - top15_subcategories_over_time.csv")
print(" - top15_subcategories_over_time.png")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 8))

im = plt.imshow(
    plot_df.T,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(im, label="Number of Papers")

plt.yticks(
    range(len(plot_df.columns)),
    plot_df.columns
)

plt.xticks(
    range(len(plot_df.index)),
    plot_df.index,
    rotation=45
)

plt.xlabel("Year")
plt.ylabel("Subcategory")
plt.title("Top 15 Subcategories Across Years (2002–2025)")

plt.tight_layout()

plt.savefig(
    "top15_subcategories_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save heatmap matrix
plot_df.to_csv("heatmap_top15_subcategories.csv")
heatmap_long = (
    plot_df
    .reset_index()
    .melt(
        id_vars="published_year",
        var_name="subcategory",
        value_name="count"
    )
)

heatmap_long.to_csv(
    "heatmap_top15_subcategories_long.csv",
    index=False
)

Top 10 Countries × Forensic Categories

In [ ]:
import os, re, ast
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "classification_framework_final.csv"
OUTDIR = "top10_country_category_heatmap"
os.makedirs(OUTDIR, exist_ok=True)

TOP_COUNTRIES = 10
COUNT_MODE = "presence"   # "presence" or "fractional"

df = pd.read_csv(CSV_PATH)

df["published_year"] = pd.to_numeric(df["published_year"], errors="coerce")
df = df[(df["published_year"] >= 2002) & (df["published_year"] <= 2025)]

# -------------------------
# Country cleaning
# -------------------------
ALIASES = {
    "usa": "USA",
    "us": "USA",
    "u.s.": "USA",
    "u.s.a.": "USA",
    "united states": "USA",
    "united states of america": "USA",
    "uk": "United Kingdom",
    "u.k.": "United Kingdom",
    "england": "United Kingdom",
    "scotland": "United Kingdom",
    "wales": "United Kingdom",
    "great britain": "United Kingdom",
    "britain": "United Kingdom",
    "south korea": "South Korea",
    "korea": "South Korea",
    "republic of korea": "South Korea",
    "uae": "United Arab Emirates",
    "u.a.e.": "United Arab Emirates",
}

def canon_country(c):
    s = str(c).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return ALIASES.get(s, str(c).strip())

def parse_countries(x):
    if pd.isna(x) or str(x).strip() == "":
        return []

    s = str(x).strip()

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return [canon_country(c) for c in parsed if str(c).strip()]
        if isinstance(parsed, dict):
            vals = parsed.get("author_countries", parsed.get("countries", []))
            return [canon_country(c) for c in vals if str(c).strip()]
    except Exception:
        pass

    # fallback for comma-separated countries
    return [canon_country(c) for c in re.split(r",|;", s) if c.strip()]

# -------------------------
# Clean category
# -------------------------
df["category"] = df["category"].astype(str).str.strip()
df = df[(df["category"] != "") & (df["category"].str.lower() != "nan")]

df["country_list"] = df["author_countries"].apply(parse_countries)
df["country_list"] = df["country_list"].apply(lambda x: sorted(set(x)))

# -------------------------
# Expand paper-country-category rows
# -------------------------
rows = []

for _, r in df.iterrows():
    countries = r["country_list"]
    category = r["category"]

    if not countries:
        continue

    weight = 1 / len(countries) if COUNT_MODE == "fractional" else 1

    for country in countries:
        rows.append({
            "country": country,
            "category": category,
            "weight": weight
        })

expanded = pd.DataFrame(rows)

# -------------------------
# Top 10 countries
# -------------------------
country_totals = (
    expanded.groupby("country")["weight"]
    .sum()
    .sort_values(ascending=False)
)

top_countries = country_totals.head(TOP_COUNTRIES).index.tolist()
expanded = expanded[expanded["country"].isin(top_countries)]

# -------------------------
# Country × category matrix
# -------------------------
matrix = (
    expanded.groupby(["country", "category"])["weight"]
    .sum()
    .unstack(fill_value=0)
    .reindex(top_countries)
)

# Convert to percentage share within each country
matrix_pct = matrix.div(matrix.sum(axis=1), axis=0) * 100

# Save CSV
matrix.to_csv(os.path.join(OUTDIR, "top10_country_category_counts.csv"))
matrix_pct.to_csv(os.path.join(OUTDIR, "top10_country_category_percent.csv"))

# -------------------------
# Heatmap
# -------------------------
fig, ax = plt.subplots(figsize=(14, 7))

im = ax.imshow(matrix_pct.values, aspect="auto")

ax.set_yticks(range(len(matrix_pct.index)))
ax.set_yticklabels(matrix_pct.index)

ax.set_xticks(range(len(matrix_pct.columns)))
ax.set_xticklabels(matrix_pct.columns, rotation=45, ha="right")

for i in range(matrix_pct.shape[0]):
    for j in range(matrix_pct.shape[1]):
        val = matrix_pct.values[i, j]
        ax.text(
            j, i, f"{val:.0f}",
            ha="center", va="center",
            fontsize=8,
            color="white" if val >= 18 else "black"
        )

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Share of country output (%)")

ax.set_title("Top 10 Countries × Forensic Categories in DFRWS (2002–2025)")
ax.set_xlabel("Forensic Category")
ax.set_ylabel("Country")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTDIR, "top10_country_category_heatmap.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(f" - {OUTDIR}/top10_country_category_counts.csv")
print(f" - {OUTDIR}/top10_country_category_percent.csv")
print(f" - {OUTDIR}/top10_country_category_heatmap.png")

TOOLS

In [ ]:
# -------------------------- Imports & Config --------------------------
import sys, subprocess
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("tools_new_expanded.xlsx") 
ENCODING = "latin-1"
SHEET_NAME = 0
SAVE_OUTPUTS = False
AUTO_INSTALL_OPENPYXL = True  

# -------------------------- Helpers --------------------------
def ensure_openpyxl():
    try:
        import openpyxl  
        return True
    except ImportError:
        if not AUTO_INSTALL_OPENPYXL:
            return False
        try:
            print("openpyxl not found. Installing...", flush=True)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
            import openpyxl  # noqa: F401
            print("openpyxl installed.\n", flush=True)
            return True
        except Exception as e:
            print(f"Failed to install openpyxl: {e}", flush=True)
            return False

def load_table(p: Path) -> pd.DataFrame:
    """
    Try CSV first. If it looks like Excel or CSV parse fails,
    try Excel (using openpyxl). Reads everything as string.
    """
    # Try CSV
    try:
        return pd.read_csv(p, encoding=ENCODING, dtype=str)
    except Exception as csv_err:
        # Fall back to Excel if CSV failed
        ok = ensure_openpyxl()
        if not ok:
            raise ImportError(
                "Missing 'openpyxl' needed to read Excel. "
                "Install it via: pip install openpyxl"
            ) from csv_err
        return pd.read_excel(p, sheet_name=SHEET_NAME, dtype=str)

def norm(s):
    if pd.isna(s):
        return None
    return str(s).strip()

def norm_lower(s):
    s = norm(s)
    return s.lower() if s is not None else None

def normalize_license(val):
    v = norm_lower(val)
    if v is None or v in {"", "na", "n/a", "none", "null"}:
        return "not-specified"
    if v.startswith("open-source"):
        return "open-source"
    if v.startswith("proprietary"):
        return "proprietary"
    if v == "not-specified":
        return "not-specified"
    if any(x in v for x in ["gpl", "mit", "apache", "bsd", "lgpl", "agpl"]):
        return "open-source"
    return "not-specified"

def normalize_action(val):
    v = norm_lower(val)
    return v if v in {"used", "created", "extended"} else None

def normalize_origin(val):
    v = norm_lower(val)
    allowed = {
        "academic_research_dfrws",
        "academic_research_external",
        "non-academia",
        "not-specified",
    }
    return v if v in allowed else "not-specified"

def map_case_insensitive(df: pd.DataFrame):
    seen = {}
    for c in df.columns:
        cl = c.strip().lower()
        if cl not in seen:
            seen[cl] = c
    return seen

def get_col(df: pd.DataFrame, colmap: dict, name: str):
    return colmap.get(name.strip().lower())

def print_table(title, table):
    print(f"\n{title}")
    print(table.to_string())

def print_counts(title, series: pd.Series):
    print(f"\n{title}")
    counts = series.value_counts(dropna=False)
    for k, v in counts.items():
        label = "NaN" if pd.isna(k) else str(k)
        print(f"  {label}: {int(v)}")

def main():
    df = load_table(INPUT_PATH)

    # Clean header names and map case-insensitively
    df.columns = [c.strip() for c in df.columns]
    colmap = map_case_insensitive(df)

    action_col  = get_col(df, colmap, "Action")
    origin_col  = get_col(df, colmap, "Origin")
    license_col = get_col(df, colmap, "License")

    # Fallback columns if missing
    if action_col is None:
        df["__Action__"] = None
        action_col = "__Action__"
    if origin_col is None:
        df["__Origin__"] = "not-specified"
        origin_col = "__Origin__"
    if license_col is None:
        df["__License__"] = "not-specified"
        license_col = "__License__"

    # Normalized columns
    df["action_norm"]  = df[action_col].apply(normalize_action)
    df["origin_norm"]  = df[origin_col].apply(normalize_origin)
    df["license_norm"] = df[license_col].apply(normalize_license)

    # Global stats
    total_rows = len(df)
    null_count = int(df["action_norm"].isna().sum())
    df_act = df[df["action_norm"].notna()].copy()
    total_action_instances = len(df_act)

    print(f"Total tool entries (paper–tool instances): {total_rows}")
    print(f"Null tool entries (no valid action): {null_count}\n")

    # Overall action breakdown
    print_counts("Overall license counts (normalized):", df["license_norm"])
    print_counts("Overall origin counts (normalized):",  df["origin_norm"])

    # Action-specific subsets
    df_used    = df[df["action_norm"] == "used"]
    df_created = df[df["action_norm"] == "created"]
    df_extended= df[df["action_norm"] == "extended"]

    print(f"\nUsed tools: {len(df_used)}")
    print(f"Created tools: {len(df_created)}")
    print(f"Extended tools: {len(df_extended)}")

    # Created breakdowns
    print_counts("Created tools by origin:",  df_created["origin_norm"])
    print_counts("Created tools by license:", df_created["license_norm"])

    df_dfrws_created = df_created[df_created["origin_norm"] == "academic_research_dfrws"]
    print_counts("Licenses of DFRWS-created tools (normalized):", df_dfrws_created["license_norm"])

    # Used breakdowns
    print_counts("Used tools by origin:",  df_used["origin_norm"])
    print_counts("Used tools by license:", df_used["license_norm"])

    # Extended breakdowns
    print_counts("Extended tools by origin:",  df_extended["origin_norm"])
    print_counts("Extended tools by license:", df_extended["license_norm"])

    # Pivot tables
    action_cols = ["used", "created", "extended"]
    if total_action_instances == 0:
        print("\nNo valid action rows found. Check the 'Action' column values.")
        return

    license_action = pd.crosstab(
        df_act["license_norm"], df_act["action_norm"], dropna=False
    ).reindex(columns=action_cols, fill_value=0)
    license_action.loc["TOTAL"] = license_action.sum()
    license_action["TOTAL"] = license_action.sum(axis=1)

    origin_action = pd.crosstab(
        df_act["origin_norm"], df_act["action_norm"], dropna=False
    ).reindex(columns=action_cols, fill_value=0)
    origin_action.loc["TOTAL"] = origin_action.sum()
    origin_action["TOTAL"] = origin_action.sum(axis=1)

    print_table("License × Action counts", license_action)
    print_table("Origin × Action counts", origin_action)

    if SAVE_OUTPUTS:
        license_action.to_csv("license_x_action_counts.csv", index=True)
        origin_action.to_csv("origin_x_action_counts.csv", index=True)
        print("\nSaved: license_x_action_counts.csv, origin_x_action_counts.csv")

if __name__ == "__main__":
    main()


Frequency 

In [ ]:
"""
Frequency of widely used tools from tools_new_expanded.csv.xlsx

What it does:
- Reads the Excel file (expects columns: Title, Tool Name, Action, Repository Link, License, Origin)
- Normalizes tool names (trim whitespace; optional lowercasing)
- Computes:
  1) Overall tool frequency (all actions)
  2) "Widely used" tool frequency (Action == "used")
  3) (Optional) Frequency by action type
- Saves results to CSVs and prints top N to console
"""

from pathlib import Path
import pandas as pd

# ----------------------------
# Config
# ----------------------------
INPUT_PATH = Path("tools_new_expanded.xlsx")  # adjust path if needed
SHEET_NAME = 0  # or set to a sheet name string
TOP_N = 30

# If you want case-insensitive merging of tool names, set True
LOWERCASE_TOOLNAMES = False

# ----------------------------
# Load
# ----------------------------
df = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME)

# Ensure expected columns exist (rename here if your headers differ slightly)
expected = ["Title", "Tool Name", "Action", "Repository Link", "License", "Origin"]
missing = [c for c in expected if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}\nFound: {list(df.columns)}")

# ----------------------------
# Clean / normalize
# ----------------------------
df["Tool Name"] = df["Tool Name"].astype("string").str.strip()
df["Action"] = df["Action"].astype("string").str.strip().str.lower()

# Drop null/empty tool names
df = df[df["Tool Name"].notna() & (df["Tool Name"] != "")].copy()

# Optional: normalize tool names to lower-case for grouping
if LOWERCASE_TOOLNAMES:
    df["Tool Name Norm"] = df["Tool Name"].str.lower()
else:
    df["Tool Name Norm"] = df["Tool Name"]

# ----------------------------
# 1) Overall frequency (all actions)
# ----------------------------
overall = (
    df.groupby("Tool Name Norm", dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values(["count", "Tool Name Norm"], ascending=[False, True])
)

# ----------------------------
# 2) Widely used = Action == "used"
# ----------------------------
used_df = df[df["Action"] == "used"].copy()

used_freq = (
    used_df.groupby("Tool Name Norm", dropna=False)
          .size()
          .reset_index(name="count_used")
          .sort_values(["count_used", "Tool Name Norm"], ascending=[False, True])
)

# ----------------------------
# 3) Frequency by action (used/created/extended) per tool
# ----------------------------
by_action = (
    df.pivot_table(
        index="Tool Name Norm",
        columns="Action",
        values="Title",
        aggfunc="size",
        fill_value=0,
        dropna=False,
    )
    .reset_index()
)

# Add totals and sort by "used" first if present
if "used" in by_action.columns:
    by_action["TOTAL"] = by_action.drop(columns=["Tool Name Norm"]).sum(axis=1)
    by_action = by_action.sort_values(["used", "TOTAL", "Tool Name Norm"], ascending=[False, False, True])
else:
    by_action["TOTAL"] = by_action.drop(columns=["Tool Name Norm"]).sum(axis=1)
    by_action = by_action.sort_values(["TOTAL", "Tool Name Norm"], ascending=[False, True])

# ----------------------------
# Output
# ----------------------------
out_dir = Path("tool_frequency_outputs")
out_dir.mkdir(exist_ok=True)

overall_path = out_dir / "tool_frequency_overall.csv"
used_path = out_dir / "tool_frequency_used_only.csv"
by_action_path = out_dir / "tool_frequency_by_action.csv"

overall.to_csv(overall_path, index=False)
used_freq.to_csv(used_path, index=False)
by_action.to_csv(by_action_path, index=False)

print("\nTop tools (overall):")
print(overall.head(TOP_N).to_string(index=False))

print("\nTop tools (Action == 'used'):")
print(used_freq.head(TOP_N).to_string(index=False))

print(f"\nSaved:\n- {overall_path}\n- {used_path}\n- {by_action_path}")

# ----------------------------
# Optional: quick bar chart for top used tools
# Uncomment if you want a plot saved as PNG
# ----------------------------
# import matplotlib.pyplot as plt
# top_used = used_freq.head(TOP_N)
# plt.figure()
# plt.bar(top_used["Tool Name Norm"], top_used["count_used"])
# plt.xticks(rotation=75, ha="right")
# plt.ylabel("Count (used)")
# plt.title(f"Top {TOP_N} Widely Used Tools (Action='used')")
# plt.tight_layout()
# plot_path = out_dir / f"top_{TOP_N}_used_tools.png"
# plt.savefig(plot_path, dpi=200)
# print(f"Saved plot: {plot_path}")


In [ ]:
"""
Complete script: Overall tool frequency (ALL actions) + bar chart with license legend
Fixes the "nan tool" issue by removing missing/blank tool names BEFORE string conversion.

Input:  tools_new_expanded.csv.xlsx
Cols:   Title, Tool Name, Action, Repository Link, License, Origin
Output:
  - Prints top N tools (overall frequency)
  - Displays bar chart (colored by license)
  - Saves cleaned frequency table + chart to disk
"""

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------
INPUT_FILE = Path("tools_new_expanded.xlsx")
TOP_N = 15

OUT_DIR = Path("tool_frequency_outputs")
OUT_DIR.mkdir(exist_ok=True)

FREQ_CSV = OUT_DIR / "tool_frequency_overall.csv"
PLOT_PNG = OUT_DIR / f"top_{TOP_N}_tools_overall_by_license.png"

LICENSE_COLORS = {
    "open-source": "#4daf4a",     # green
    "proprietary": "#e41a1c",     # red
    "not-specified": "#999999",   # gray
}

# Canonical tool-name normalization rules
# (Add more ecosystems here as needed)
TOOL_NORMALIZATION = [
    # Normalize all Cellebrite variants to an ecosystem label
    (r"\bcellebrite\b", "Cellebrite UFED (ecosystem)"),
]

# ----------------------------
# Load
# ----------------------------
df = pd.read_excel(INPUT_FILE)

# ----------------------------
# Clean (IMPORTANT: drop NaN tool names before converting to string)
# ----------------------------
df = df[df["Tool Name"].notna()].copy()

df["Tool Name"] = df["Tool Name"].astype("string").str.strip()
df["Action"] = df["Action"].astype("string").str.lower().str.strip()
df["License"] = df["License"].astype("string").str.lower().str.strip()

# Drop empty tool names and any accidental "nan" strings
df = df[df["Tool Name"].notna() & (df["Tool Name"] != "")]
df = df[df["Tool Name"].str.lower() != "nan"]

# Exclude DB Browser for SQLite
df = df[df["Tool Name"].str.lower() != "db browser for sqlite"]

# ----------------------------
# Normalize tool names (ecosystem-level)
# ----------------------------
for pattern, canonical in TOOL_NORMALIZATION:
    df.loc[df["Tool Name"].str.contains(pattern, case=False, na=False, regex=True), "Tool Name"] = canonical

# ----------------------------
# Normalize License values (optional safety)
# ----------------------------
df["License"] = df["License"].fillna("not-specified")
df.loc[~df["License"].isin(LICENSE_COLORS.keys()), "License"] = "not-specified"

# ----------------------------
# Overall frequency (all actions)
# ----------------------------
freq = (
    df.groupby(["Tool Name", "License"], dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values(["count", "Tool Name"], ascending=[False, True])
)

# Save frequency table
freq.to_csv(FREQ_CSV, index=False)

# ----------------------------
# Display table (top N)
# ----------------------------
print(f"\nTop {TOP_N} tools by overall frequency (all actions):\n")
print(freq.head(TOP_N).to_string(index=False))

# ----------------------------
# Plot (top N)
# ----------------------------
top = freq.head(TOP_N).copy()
colors = top["License"].map(lambda x: LICENSE_COLORS.get(x, LICENSE_COLORS["not-specified"]))

plt.figure(figsize=(11, 6))
plt.bar(top["Tool Name"], top["count"], color=colors)
plt.xticks(rotation=70, ha="right")
plt.ylabel("Frequency (all actions)")
plt.title(f"Top {TOP_N} Tools by Overall Frequency (colored by license)")

# Legend
legend_handles = [
    plt.Line2D([0], [0], color=c, lw=6, label=l.replace("-", " ").title())
    for l, c in LICENSE_COLORS.items()
]
plt.legend(handles=legend_handles, title="License", frameon=False)

plt.tight_layout()
plt.savefig(PLOT_PNG, dpi=300)
plt.show()

print(f"\nSaved:\n- Frequency table: {FREQ_CSV}\n- Plot: {PLOT_PNG}")


CREATED-TOOL ANALYSIS

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("Tools_only_created.csv")

# Read second header row
df = pd.read_csv(INPUT_PATH, header=1)
df.columns = df.columns.astype(str).str.strip()

label_columns = [
    "Hosting Location",
    "Distribution Format",
    "Accessibility/Availability Status",
    "Maintenance Status",
    "License",
]

TOTAL = len(df)

for col in label_columns:

    print("\n" + "=" * 70)
    print(f"{col.upper()}")
    print("=" * 70)

    series = (
        df[col]
        .astype("string")
        .str.strip()
        .fillna("missing")
        .replace("", "missing")
    )

    counts = series.value_counts()
    percents = (counts / TOTAL * 100).round(2)

    result = pd.DataFrame({
        "Label": counts.index,
        "Count": counts.values,
        "Percentage (%)": percents.values
    })

    print(result.to_string(index=False))

    print("-" * 70)
    print(f"TOTAL TOOLS: {TOTAL}")